In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 12.1
Device count: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import pandas as pd
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import torch.nn as nn

from torch.utils.data import DataLoader
from sklearn.neighbors import kneighbors_graph

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# Display all columns
pd.set_option('display.max_columns', None)

# Display all rows
pd.set_option('display.max_rows', None)

# Set the width to show all content in each cell
pd.set_option('display.width', None)

# Set the max string length to display
pd.set_option('display.max_colwidth', None)

In [5]:
sys.path.append('C:/Users/rishe/Dissertation')

In [6]:
EXP_ID = f'exp_12_gcn_gru_baseline'

In [7]:
# Setup paths
model_save_path = f"C:/Users/rishe/Dissertation/experiments/saved_models/{EXP_ID}"
log_save_path = f"C:/Users/rishe/Dissertation/experiments/logs/{EXP_ID}"

print(f"Model save path: {model_save_path}")
print(f"Log save path: {log_save_path}")

Model save path: C:/Users/rishe/Dissertation/experiments/saved_models/exp_12_gcn_gru_baseline
Log save path: C:/Users/rishe/Dissertation/experiments/logs/exp_12_gcn_gru_baseline


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [9]:
# ====== CONFIG ======

SEQ_LEN = 7
HORIZON = 1
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
CONFIG = {
    "epochs": 30,
    "lr": 1e-3,
    "batch_size": 32,
    "hidden_dim": 64,
    "criterion": torch.nn.MSELoss(),
    "device": device,
    "save_dir": model_save_path,
    "log_dir": log_save_path,
}

#### setup cell

In [10]:
# ====== SETUP ======

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(device)

cuda


#### Data pipeline

In [11]:
from utils.data_utils.data_helper_utils import *

In [12]:
# ===== LOAD PIVOTS =====
pivot_path = "C:/Users/rishe/Dissertation/data/era5_pivot_data/"

pivots = load_pivots(pivot_path)

print("Loaded pivots:", list(pivots.keys()))

Loaded pivots: ['rain', 'temp', 'dew', 'pressure', 'u10', 'v10']


In [13]:
# ===== LOAD STATION METADATA =====

station_df = pd.read_csv(
    "C:/Users/rishe/Dissertation/data/wb_station_coords.csv"
).rename(columns={
    "latitude": "lat",
    "longitude": "lon"
})

print(station_df.head())

        station_id        lat        lon
0         AKRIGANJ  24.310000  88.360000
1          ALGARAH  27.120000  88.580000
2           ALIPUR  22.530000  88.330000
3       ALIPURDUAR  26.470000  89.550000
4  ALIPURDUAR(CWC)  26.498333  89.528333


In [14]:
# ===== GRAPH CONSTRUCTION =====
lat, lon = get_lat_lon_aligned(pivots['rain'], station_df)

edge_index = build_edge_index_radius(lat, lon, threshold_km=100)
edge_index = edge_index.to(device)

print("Edge index shape:", edge_index.shape)

Edge index shape: torch.Size([2, 18110])


In [15]:
# ===== SCALING =====

scaled_pivots, scalers = scale_pivots(pivots)

print("Scaling done ✅")

Scaling done ✅


In [16]:
from utils.data_utils.dataset_files.gnn_dataset import *

In [17]:
use_latent = True  #  toggle here

X, feature_order = build_feature_tensor(scaled_pivots, use_latent)  # ✅ Unpack the tuple

print("X shape:", X.shape)

X shape: (19723, 291, 6)


In [18]:
time_feats = build_time_features(pivots['rain'].index)
print("Time features shape:", time_feats.shape)

Time features shape: (19723, 6)


In [19]:
X = add_time_features(X, time_feats)
print("Final X shape:", X.shape)

Final X shape: (19723, 291, 12)


In [20]:

X_train, X_val, X_test = temporal_split(
    X, pivots['rain'].index
)

print(X_train.shape, X_val.shape, X_test.shape)

(13806, 291, 12) (2958, 291, 12) (2959, 291, 12)


In [21]:

train_ds = SpatioTemporalDataset(X_train)
val_ds   = SpatioTemporalDataset(X_val)
test_ds  = SpatioTemporalDataset(X_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

print("DataLoaders ready ✅")

DataLoaders ready ✅


#### sanity checks

In [22]:
# ====== SANITY CHECK ======

x, y = next(iter(train_loader))

print("X shape:", x.shape)  # (B, L, N, F)
print("Y shape:", y.shape)  # (B, N)

print("Sample stats:")
print("Mean:", x.mean().item(), "Std:", x.std().item())

X shape: torch.Size([32, 7, 291, 12])
Y shape: torch.Size([32, 291])
Sample stats:
Mean: 0.0595509298145771 Std: 0.8387560844421387


#### model import

In [23]:
from models.gcn_gru import GCN_GRU_Model, train_model

In [24]:
# ====== INIT MODEL ======

sample_x, _ = next(iter(train_loader))
_, _, N, F = sample_x.shape

In [25]:
model_type = EXP_ID
experiment_name = f"{model_type}_{'latent' if use_latent else 'rain'}"
print(f"Experiment name: {experiment_name}")

Experiment name: exp_12_gcn_gru_baseline_latent


In [31]:
model = GCN_GRU_Model(
    in_channels=F,
    hidden_dim=CONFIG["hidden_dim"]
)

In [32]:
model = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    model=model,
    edge_index=edge_index,
    device=CONFIG["device"],
    epochs=CONFIG["epochs"],
    lr=CONFIG["lr"],
    criterion=CONFIG["criterion"],
    save_dir=CONFIG["save_dir"],
    log_dir=CONFIG["log_dir"],
    experiment_name=EXP_ID
)

2026-03-20 12:26:21 | INFO | Starting GCN+GRU training
2026-03-20 12:26:21 | INFO | Device: cuda
2026-03-20 12:26:21 | INFO | Experiment Config:
2026-03-20 12:26:21 | INFO | LR: 0.001
2026-03-20 12:26:21 | INFO | Epochs: 30
2026-03-20 12:26:21 | INFO | Criterion: MSELoss()


Train:   0%|          | 0/432 [00:00<?, ?it/s]

Eval:   0%|          | 0/93 [00:00<?, ?it/s]

2026-03-20 12:39:00 | INFO | Epoch 001 | Train: 0.633666 | Val: 0.559077
2026-03-20 12:39:00 | INFO | Checkpoint saved: C:/Users/rishe/Dissertation/experiments/saved_models/exp_12_gcn_gru_baseline\epoch_1.pt
2026-03-20 12:39:00 | INFO | ✅ New best model saved: C:/Users/rishe/Dissertation/experiments/saved_models/exp_12_gcn_gru_baseline\exp_12_gcn_gru_baseline_best.pt


Train:   0%|          | 0/432 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/93 [00:00<?, ?it/s]

2026-03-20 14:10:27 | INFO | Epoch 029 | Train: 0.500719 | Val: 0.553091
2026-03-20 14:10:27 | INFO | Checkpoint saved: C:/Users/rishe/Dissertation/experiments/saved_models/exp_12_gcn_gru_baseline\epoch_29.pt


Train:   0%|          | 0/432 [00:00<?, ?it/s]

Eval:   0%|          | 0/93 [00:00<?, ?it/s]

2026-03-20 14:11:17 | INFO | Epoch 030 | Train: 0.495355 | Val: 0.561164
2026-03-20 14:11:17 | INFO | Checkpoint saved: C:/Users/rishe/Dissertation/experiments/saved_models/exp_12_gcn_gru_baseline\epoch_30.pt
2026-03-20 14:11:17 | INFO | Training completed


##### Load saved model

In [26]:
# ===== MODEL RECONSTRUCTION =====

sample_x, _ = next(iter(test_loader))
_, _, N, F = sample_x.shape

model = GCN_GRU_Model(
    in_channels=F,
    hidden_dim=64   # MUST match training
).to(device)

print("Model rebuilt with F =", F)

Model rebuilt with F = 12


In [27]:
model_save_path

'C:/Users/rishe/Dissertation/experiments/saved_models/exp_12_gcn_gru_baseline'

In [29]:
# ===== LOAD CHECKPOINT =====

ckpt_path = f"{model_save_path}/exp_12_gcn_gru_baseline_best.pt"

checkpoint = torch.load(ckpt_path, map_location=device)

# case 1: full checkpoint dict
if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    # case 2: only state_dict
    model.load_state_dict(checkpoint)

model.eval()

print("Checkpoint loaded ✅")

Checkpoint loaded ✅


C:\Users\rishe\AppData\Local\Temp\ipykernel_64816\512938328.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


In [30]:
# ===== COLLECT PREDICTIONS =====

preds_all = []
targets_all = []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        preds = model(x, edge_index)

        preds_all.append(preds.cpu().numpy())
        targets_all.append(y.cpu().numpy())

preds = np.concatenate(preds_all)   # (num_samples, N)
targets = np.concatenate(targets_all)

print("Predictions shape:", preds.shape)

Predictions shape: (2952, 291)


#### Metrics

In [34]:
# ===== BACK-SCALE =====

rain_mu, rain_sigma = scalers['rain']

preds_real = preds * rain_sigma + rain_mu
targets_real = targets * rain_sigma + rain_mu

print("Back-scaling done ✅")

Back-scaling done ✅


In [ ]:
# ===== OVERALL METRICS =====

from utils.metric_utils.metrics import rmse, mae, bias, nrmse

def compute_metrics(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mae(y_true, y_pred),
        "Bias": bias(y_true, y_pred),
        "NRMSE": nrmse(y_true, y_pred),
    }

# flatten
preds_flat = preds.reshape(-1)
targets_flat = targets.reshape(-1)

preds_real_flat = preds_real.reshape(-1)
targets_real_flat = targets_real.reshape(-1)
t = 0.23
print("\n===== SCALED METRICS =====")
scaled_results = compute_metrics(targets_flat, preds_flat)
for k, v in scaled_results.items():
    print(f"{k}: {v:.4f}")

print("\n===== REAL METRICS =====")
real_results = compute_metrics(targets_real_flat, preds_real_flat)
for k, v in real_results.items():
    print(f"{k}: {v:.4f}")


===== SCALED METRICS =====
RMSE: 0.7168
MAE: 0.3253
Bias: 0.0135
NRMSE: 85.3869

===== REAL METRICS =====
RMSE: 6.9229
MAE: 3.1413
Bias: 0.1307
NRMSE: 1.2831


In [31]:
# ===== METRICS =====

from utils.metric_utils.metrics import rmse, mae, bias, nrmse

preds_flat = preds.reshape(-1)
targets_flat = targets.reshape(-1)

results = {
    "RMSE": rmse(targets_flat, preds_flat),
    "MAE": mae(targets_flat, preds_flat),
    "Bias": bias(targets_flat, preds_flat),
    "NRMSE": nrmse(targets_flat, preds_flat),
}

print("\n===== TEST METRICS =====")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


===== TEST METRICS =====
RMSE: 0.7168
MAE: 0.3253
Bias: 0.0135
NRMSE: 85.3869


#### Seasonal metrics

In [32]:
# ===== SEASONAL METRICS =====

dates_test = pivots['rain'].index[-len(X_test):]

months = dates_test.month.values

# adjust for sliding window
months = months[7:]  # seq_len = 7

# expand to match (samples, N)
months_expanded = np.repeat(months[:, None], preds.shape[1], axis=1)
months_flat = months_expanded.reshape(-1)

seasons = {
    "monsoon": [6, 7, 8, 9],
    "non_monsoon": [1, 2, 3, 4, 5, 10, 11, 12],
}

for name, season_months in seasons.items():
    mask = np.isin(months_flat, season_months)

    season_rmse = rmse(
        targets_flat[mask],
        preds_flat[mask]
    )

    print(f"{name.upper()} RMSE: {season_rmse:.4f}")

MONSOON RMSE: 1.0514
NON_MONSOON RMSE: 0.4707


In [ ]:
# ===== SEASONAL METRICS =====

dates_test = pivots['rain'].index[-len(X_test):]
months = dates_test.month.values

# adjust for sequence length
seq_len = 7
months = months[seq_len:]

# expand to match (samples, N)
months_expanded = np.repeat(months[:, None], preds.shape[1], axis=1)
months_flat = months_expanded.reshape(-1)

seasons = {
    "monsoon": [6, 7, 8, 9],
    "non_monsoon": [1, 2, 3, 4, 5, 10, 11, 12],
}

print("\n===== SEASONAL METRICS (REAL SCALE) =====")

for name, season_months in seasons.items():
    mask = np.isin(months_flat, season_months)

    res = compute_metrics(
        targets_real_flat[mask],
        preds_real_flat[mask]
    )

    print(f"\n{name.upper()}")
    for k, v in res.items():
        print(f"{k}: {v:.4f}")


===== SEASONAL METRICS (REAL SCALE) =====

MONSOON
RMSE: 10.1546
MAE: 6.0526
Bias: 0.9290
NRMSE: 0.8567

NON_MONSOON
RMSE: 4.5461
MAE: 1.7033
Bias: -0.2635
NRMSE: 2.0611


#### Per station metrics

In [39]:
# ===== PER-STATION METRICS =====

N = preds.shape[1]

station_metrics = []

for i in range(N):
    y_true = targets_real[:, i]
    y_pred = preds_real[:, i]

    res = compute_metrics(y_true, y_pred)

    res["station_idx"] = i
    station_metrics.append(res)

station_df_metrics = pd.DataFrame(station_metrics)

print(station_df_metrics.head())

        RMSE       MAE      Bias     NRMSE  station_idx
0   6.483563  3.403525  0.270861  1.582707            0
1  12.916238  5.947776 -0.117874  1.355267            1
2   7.602301  3.503347 -0.243575  1.668330            2
3  13.995602  6.326431  0.876301  1.544420            3
4  14.003246  6.317963  0.837251  1.545263            4


In [40]:
# ===== ADD STATION IDS =====

station_ids = pivots['rain'].columns.tolist()

station_df_metrics["station_id"] = station_ids

station_df_metrics = station_df_metrics[[
    "station_id", "RMSE", "MAE", "Bias", "NRMSE"
]]

station_df_metrics.sort_values("RMSE").head()

,station_id,RMSE,MAE,Bias,NRMSE
48,BURNPUR,5.388258,2.998229,0.448254,1.510604
12,ASANSOL,5.388260,2.998230,0.448254,1.510605
280,TANTLOI,5.392615,2.975432,0.572601,1.494744
13,ASANSOLE %A,5.398691,2.961021,0.457602,1.517506
178,MAJHIA,5.415776,2.978952,0.315182,1.460128


In [41]:
# ===== BEST / WORST =====

print("\nBest stations:")
display(station_df_metrics.nsmallest(5, "RMSE"))

print("\nWorst stations:")
display(station_df_metrics.nlargest(5, "RMSE"))


Best stations:


,station_id,RMSE,MAE,Bias,NRMSE
48,BURNPUR,5.388258,2.998229,0.448254,1.510604
12,ASANSOL,5.388260,2.998230,0.448254,1.510605
280,TANTLOI,5.392615,2.975432,0.572601,1.494744
13,ASANSOLE %A,5.398691,2.961021,0.457602,1.517506
178,MAJHIA,5.415776,2.978952,0.315182,1.460128



Worst stations:


,station_id,RMSE,MAE,Bias,NRMSE
50,BUXADUR,16.020041,6.865354,-0.231045,1.576210
166,KUMARGRAM,15.938281,6.808636,0.208085,1.615869
49,BUXA,15.935489,6.823342,-0.220780,1.567891
41,BIJANBARI,15.770782,7.849798,-1.349762,1.483297
270,SINGLA BAZAR,15.748839,7.837149,-1.361108,1.481233


In [46]:
result_save_path = f'C:/Users/rishe/Dissertation/experiments/results/{EXP_ID}'
os.makedirs(result_save_path, exist_ok=True)

In [47]:
# ===== SAVE =====
station_df_metrics.to_csv(f"{result_save_path}/station_metrics.csv", index=False)

pd.DataFrame(real_results, index=[0]).to_csv(f"{result_save_path}/overall_metrics.csv", index=False)

print("Saved all metrics ✅")

Saved all metrics ✅
